In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
data_annotated_path = "/home/bmb/haxx/working/ceisel_mumm/"
data = sc.read_h5ad(data_annotated_path + "joint_annotated.h5ad")
data.shape

# Building A Timeline

In [ ]:
plt.figure()
plt.title("Log total counts vs time")
plt.boxplot(
    [data.obs['log_total_counts'][data.obs['time'] == t] for t in [12,24,48,72]],
    showfliers=False,
)
plt.xticks(
    [1,2,3,4],
    ["12h","24h","48h","72h"]
)
plt.ylabel("Log total counts")
plt.show()


In [ ]:
# PCA IQR Timeline, Big Sheet

In [ ]:
control_mask =  data.obs['exp_condition'] == "Cntr"
mutant_mask = data.obs['exp_condition'] == "Mtz"

times = [12,24,48,72]

time_masks = [
    data.obs['time'] == t
    for t in times
]

fig,axes = plt.subplots(5,10,figsize=(30,15))
for i in range(50):

    time_control_iqrs = {}
    time_mutant_iqrs = {}
    for t,time_mask in zip(times,time_masks):
        combined_mask = control_mask & time_mask
        time_control_iqrs[t] = {
            'low':np.percentile(data.obsm['X_pca'][:,i][combined_mask],25),
            'median':np.median(data.obsm['X_pca'][:,i][combined_mask]),
            'high':np.percentile(data.obsm['X_pca'][:,i][combined_mask],75)
        }
        time_mutant_iqrs[t] = {
            'low':np.percentile(data.obsm['X_pca'][:,i][mutant_mask & time_mask],25),
            'median':np.median(data.obsm['X_pca'][:,i][mutant_mask & time_mask]),
            'high':np.percentile(data.obsm['X_pca'][:,i][mutant_mask & time_mask],75)
        }
    
    ax = axes[i//10,i%10]
    ax.plot(
        times,
        [time_control_iqrs[t]['median'] for t in times],
        label="control"
    )
    ax.fill_between(
        times,
        [time_control_iqrs[t]['low'] for t in times],
        [time_control_iqrs[t]['high'] for t in times],
        alpha=.2
    )
    ax.plot(
        times,
        [time_mutant_iqrs[t]['median'] for t in times],
        label="mutant"
    )
    ax.fill_between(
        times,
        [time_mutant_iqrs[t]['low'] for t in times],
        [time_mutant_iqrs[t]['high'] for t in times],
        alpha=.2
    )
    ax.set_title(f"PC{i+1}")
    ax.legend()
    # ax.plot(
    #     control_data.obs['time'],
    #     control_data.obsm['X_pca'][:,i],
    #     label="control"
    # )
    # ax.plot(
    #     mutant_data.obs['time'],
    #     mutant_data.obsm['X_pca'][:,i],
    #     label="mutant"
    # )
    # ax.legend()
fig.tight_layout()
fig.show()

# Leiden 

In [ ]:
sc.tl.leiden(data, resolution=0.5)

In [ ]:
# sc.pl.umap(data, color=['leiden'],legend_loc='on data')

In [ ]:
sc.tl.rank_genes_groups(data, 'leiden')
sc.pl.rank_genes_groups(data, n_genes=25)


In [ ]:
top_3_across_all = data.uns['rank_genes_groups']['names'][:3]
top_3_across_all = np.array([[top_3_across_all[j][i] for i in range(28)] for j in range(3)])

In [ ]:
top_3_across_all.flatten()

In [ ]:
# sc.pl.dotplot(data, top_3_across_all.flatten(),groupby="leiden",figsize=(20,10))
sc.pl.dotplot(data, ['rbpms2a','rbpms2b'],groupby="leiden",figsize=(20,10))

# Projection sets 

In [ ]:
# Using neural paper with mystery meat sourcing 
signature_genes = ["Il1b","Aif1","App","Dlg4","Endog","F2","Grin2b","Icam1","Il1b","Map2k1","Mapk1","Mmp9","Parg","Parp1","Plat","S100b","Sirt1"]
signature_genes = [g.lower() for g in signature_genes]




In [ ]:
data.var_names
var_name_set = set(data.var_names)

In [ ]:
[g for g in data.var_names if "l1b" in g]

In [ ]:
set(signature_genes) - ({g for g in signature_genes if g in var_name_set})

In [ ]:
[g for g in data.var_names if "arp" in g]